# Argentina SYNOP cloud base height retrieval

Starter notebook for **cloudVis**: download surface synoptic (SYNOP) observations over Argentina and extract **cloud base height**.

**Target variable**
- **Cloud base height** (`HKm` in decoded OGIMET tables, or `cloud_base_m` / `cloud_base_ft` after decoding raw SYNOP)
- Units: meters or feet (`feet = meters × 3.28084`)

**Data sources**
1. [`atmofetch`](https://pypi.org/project/atmofetch/) / OGIMET decoded hourly tables — convenient, but often lags for the current day
2. [OGIMET raw SYNOP API](http://www.ogimet.com/cgi-bin/getsynop) — near-real-time fallback used automatically when decoded data is empty

Later steps: spatial interpolation over Argentina and optional export to AERMET-compatible formats.

In [ ]:
# Run this cell first in Google Colab
!pip install -q atmofetch pandas requests folium

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from io import StringIO
from typing import Iterable

import pandas as pd
import pymetdecoder.synop as synop_decoder
import requests
from atmofetch import meteo_ogimet, stations_ogimet

# --- user configuration ---
END_UTC = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
START_UTC = END_UTC - timedelta(hours=6)  # keep short for Colab; increase once it works

M_TO_FEET = 3.28084
KM_TO_FEET = 1000 * M_TO_FEET
OGIMET_URL = "http://www.ogimet.com/cgi-bin/getsynop"


def station_id_from_decoded(decoded_msg: dict, fallback: int | str) -> str:
    """Normalize pymetdecoder station identifiers to a WMO string."""
    station_id = decoded_msg.get("station_id", fallback)
    if isinstance(station_id, dict):
        station_id = station_id.get("value", fallback)
    return str(station_id)


def extract_cloud_base_m(decoded_msg: dict) -> float | None:
    """Return cloud base height in meters from a pymetdecoder SYNOP dict."""
    base = decoded_msg.get("lowest_cloud_base")
    if not base:
        return None
    if "value" in base:
        return float(base["value"])
    if "min" in base and base["min"] is not None:
        return float(base["min"])
    return None


def download_raw_synop_argentina(start: datetime, end: datetime) -> pd.DataFrame:
    params = {
        "begin": start.strftime("%Y%m%d%H%M"),
        "end": end.strftime("%Y%m%d%H%M"),
        "state": "Argen",
        "header": "yes",
        "lang": "eng",
    }
    response = requests.get(OGIMET_URL, params=params, timeout=120)
    response.raise_for_status()
    raw = pd.read_csv(StringIO(response.text))
    return raw.rename(
        columns={
            "WMO_ID": "station_id",
            "ANO": "year",
            "MES": "month",
            "DIA": "day",
            "HORA": "hour",
            "MINUTO": "minute",
            "PARTE": "report",
        }
    )


def decode_raw_synop_df(raw_df: pd.DataFrame, stations_df: pd.DataFrame) -> pd.DataFrame:
    """Decode raw OGIMET SYNOP bulletins into cloud-base observations."""
    records: list[dict] = []
    for _, row in raw_df.iterrows():
        try:
            decoded_msg = synop_decoder.SYNOP().decode(row["report"])
        except Exception:
            continue

        base_m = extract_cloud_base_m(decoded_msg)
        station_id = station_id_from_decoded(decoded_msg, row["station_id"])
        records.append(
            {
                "station_id": station_id,
                "datetime_utc": pd.Timestamp(
                    year=int(row["year"]),
                    month=int(row["month"]),
                    day=int(row["day"]),
                    hour=int(row["hour"]),
                    minute=int(row["minute"]),
                    tz="UTC",
                ),
                "cloud_base_m": base_m,
                "cloud_base_ft": base_m * M_TO_FEET if base_m is not None else None,
                "report": row["report"],
            }
        )

    if not records:
        return pd.DataFrame()

    decoded_df = pd.DataFrame(records)
    station_lookup = stations_df.rename(columns={"wmo_id": "station_id"}).copy()
    station_lookup["station_id"] = station_lookup["station_id"].astype(str)
    decoded_df["station_id"] = decoded_df["station_id"].astype(str)

    return decoded_df.merge(
        station_lookup[["station_id", "lat", "lon", "station_names"]],
        on="station_id",
        how="left",
    )


def normalize_observations(df: pd.DataFrame) -> pd.DataFrame:
    """Canonical schema for all downstream cells and the map."""
    if df.empty:
        return pd.DataFrame(
            columns=[
                "station_ID",
                "Date",
                "lat",
                "lon",
                "station_names",
                "cloud_base_m",
                "cloud_base_ft",
                "HKm",
                "data_source",
            ]
        )

    out = df.copy()

    if "station_ID" not in out.columns:
        if "station_id" in out.columns:
            out["station_ID"] = pd.to_numeric(out["station_id"].astype(str), errors="coerce")
        else:
            out["station_ID"] = pd.NA

    if "Date" not in out.columns:
        out["Date"] = pd.to_datetime(out["datetime_utc"], utc=True, errors="coerce")
    else:
        out["Date"] = pd.to_datetime(out["Date"], utc=True, errors="coerce")

    if "lat" not in out.columns and "Lat" in out.columns:
        out["lat"] = out["Lat"]
    if "lon" not in out.columns and "Lon" in out.columns:
        out["lon"] = out["Lon"]

    if "cloud_base_m" not in out.columns:
        if "HKm" in out.columns:
            out["cloud_base_m"] = pd.to_numeric(out["HKm"], errors="coerce") * 1000
        else:
            out["cloud_base_m"] = pd.NA

    if "cloud_base_ft" not in out.columns:
        out["cloud_base_ft"] = pd.to_numeric(out["cloud_base_m"], errors="coerce") * M_TO_FEET

    if "HKm" not in out.columns:
        out["HKm"] = pd.to_numeric(out["cloud_base_m"], errors="coerce") / 1000

    if "data_source" not in out.columns:
        out["data_source"] = "unknown"

    return out


def fetch_decoded_synop(
    station_ids: Iterable[int | str],
    start: datetime,
    end: datetime,
    stations_df: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    """Download SYNOP cloud-base data; fall back to raw OGIMET when decoded data is empty."""
    station_ids = [str(s) for s in station_ids]

    df = meteo_ogimet(
        interval="hourly",
        station=[int(s) for s in station_ids],
        date_range=(start.date().isoformat(), end.date().isoformat()),
        coords=True,
    )

    if not df.empty and "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
        mask = (df["Date"] >= pd.Timestamp(start)) & (df["Date"] <= pd.Timestamp(end))
        df = df.loc[mask].copy()
        if not df.empty:
            if "HKm" in df.columns:
                df["cloud_base_m"] = pd.to_numeric(df["HKm"], errors="coerce") * 1000
                df["cloud_base_ft"] = df["cloud_base_m"] * M_TO_FEET
            df["data_source"] = "ogimet_decoded"
            return normalize_observations(df), "ogimet_decoded"

    print("Decoded OGIMET tables are empty for this window; falling back to raw SYNOP API...")
    raw = download_raw_synop_argentina(start, end)
    raw = raw[raw["station_id"].astype(str).isin(station_ids)]
    fallback = decode_raw_synop_df(raw, stations_df)
    if fallback.empty:
        return normalize_observations(pd.DataFrame()), "none"

    mask = (fallback["datetime_utc"] >= pd.Timestamp(start)) & (
        fallback["datetime_utc"] <= pd.Timestamp(end)
    )
    fallback = fallback.loc[mask].copy()
    fallback["data_source"] = "ogimet_raw"
    return normalize_observations(fallback), "ogimet_raw"

## 1. List Argentina SYNOP stations (with coordinates)

Argentina WMO block is **87xxx** (South America).

In [ ]:
stations = stations_ogimet(country="Argentina")
print(f"{len(stations)} stations found")
stations.head(10)

## 2. Download cloud base height observations

Tries OGIMET decoded hourly tables first. If they are empty (common for **today**), automatically falls back to the raw SYNOP API and decodes cloud base height with `pymetdecoder`.

In [ ]:
sample_station_ids = stations["wmo_id"].head(8).tolist()
decoded, data_source = fetch_decoded_synop(sample_station_ids, START_UTC, END_UTC, stations)

print(f"Data source: {data_source}")
print(f"Rows: {len(decoded)}")
print(f"Rows with cloud base height: {decoded['cloud_base_ft'].notna().sum()}")

preview_cols = [
    "station_ID",
    "Date",
    "lat",
    "lon",
    "cloud_base_m",
    "cloud_base_ft",
    "data_source",
]
decoded[preview_cols].dropna(subset=["cloud_base_ft"]).head(20)

## 3. Download raw SYNOP bulletins for all Argentina (bulk)

Optional: inspect the raw messages behind the fallback path.

In [ ]:
raw = download_raw_synop_argentina(START_UTC, END_UTC)
print(f"{len(raw)} raw SYNOP messages")
raw.head()

## 4. Decode raw SYNOP messages (cloud base height)

Uses the same decoder as the fallback inside `fetch_decoded_synop`.

In [ ]:
decoded_from_raw = decode_raw_synop_df(raw, stations)
decoded_from_raw = normalize_observations(decoded_from_raw)
decoded_from_raw.sort_values("Date").head(20)

## 5. Quick sanity check: stations reporting cloud base height

In [ ]:
with_base = decoded.dropna(subset=["cloud_base_ft"]).copy()
print(f"Messages with cloud base height: {len(with_base)} / {len(decoded)}")

if not with_base.empty:
    display(
        with_base[
            [
                "Date",
                "station_ID",
                "station_names",
                "lat",
                "lon",
                "cloud_base_m",
                "cloud_base_ft",
                "data_source",
            ]
        ]
        .sort_values("cloud_base_ft")
        .head(15)
    )
else:
    print("No cloud base height in this time window. Try a longer range or more stations.")

## 6. Preview map (station points only)

Plots `decoded`, which always uses the normalized schema (`lat`, `lon`, `cloud_base_ft`). Interpolation over the full territory comes later.

In [ ]:
import folium

map_df = decoded.dropna(subset=["cloud_base_ft", "lat", "lon"]).copy()

if map_df.empty:
    print("No cloud-base observations to map in the selected window.")
    print(f"`decoded` has {len(decoded)} rows; data source was '{data_source}'.")
else:
    m = folium.Map(location=[-38.5, -63.5], zoom_start=4, tiles="CartoDB positron")
    for _, row in map_df.iterrows():
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=6,
            popup=(
                f"Station {int(row['station_ID'])}<br>"
                f"UTC: {row['Date']}<br>"
                f"Cloud base: {row['cloud_base_ft']:.0f} ft ({row['cloud_base_m']:.0f} m)<br>"
                f"Source: {row['data_source']}"
            ),
            color="#2563eb",
            fill=True,
            fill_opacity=0.8,
        ).add_to(m)
    m

## 7. Can SYNOP be converted to AERMET?

If you meant **AERMET** (EPA's meteorological preprocessor for AERMOD):

- **Not directly.** AERMET does not ingest raw SYNOP bulletins.
- **Yes, indirectly.** After decoding SYNOP into hourly surface observations, you can export to an archive format AERMET accepts, such as **ISHD/ISD**, **SAMSON**, **CD-144**, or **EXTRACT**.
- AERMET needs more than cloud base: temperature, wind, pressure, humidity, precipitation, etc. Your decoded dataframe already contains several of these when using the OGIMET decoded path.
- For **cloud base height mapping**, you do **not** need AERMET. AERMET is mainly for air-dispersion modeling workflows.

If you meant a different format (for example an internal aviation product called AEROMET), share a sample file and we can map fields explicitly.

In [ ]:
# Optional: export the normalized observations for external tools
export_cols = [
    "Date",
    "station_ID",
    "lat",
    "lon",
    "cloud_base_m",
    "cloud_base_ft",
    "data_source",
]
decoded[export_cols].to_csv("argentina_cloud_base.csv", index=False)
print("Wrote argentina_cloud_base.csv")

## Next steps

1. **Scale up stations**: loop over all `stations['wmo_id']` in batches.
2. **Pick one timestamp**: filter `decoded` to a single UTC hour before mapping/interpolating.
3. **Interpolation**: `scipy.interpolate.griddata` or `pykrige` on a lat/lon grid clipped to Argentina's border.
4. **AERMET export** (only if needed for dispersion modeling): build an ISHD/EXTRACT writer from the decoded hourly fields.

Share this notebook from Colab via *File → Share* so coworkers can run it in their browsers.